In [1]:
# Bibliotecas usadas para ler os dados, treinar e avaliar as redes
import os
import json
import joblib  # Pra salvar o normalizador (scaler)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm  # Barra de progresso

# Métricas do sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Usa uma seed fixa para facilitar a repetição dos resultados
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Conecta o treinamento com a placa de vídeo
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo usado: {DEVICE}")


Dispositivo usado: cuda


In [2]:
# Lê os melhores parâmetros encontrados pelo notebook de tuning
DIR_MODELOS = "modelos_treinados"
os.makedirs(DIR_MODELOS, exist_ok=True)

# Abre o arquivo de resumo do Optuna
with open("tuning_bayesiano/resumo_melhores_hp_binario.json", "r") as f:
    melhores_hp = json.load(f)

In [3]:
# Prepara os dados e separa um teste que fica de fora do treinamento
CENARIO = "binario"

# Leitura do dataset
df = pd.read_csv("Normal+Attacks.csv")

# X contém as entradas numéricas; y guarda a classe que queremos prever
X = df.drop(columns=["label"], errors="ignore")
X = X.select_dtypes(include="number")
# Trata o y para separar de forma binária em ataque ou não ataque
y = df["label"].apply(lambda x: 0 if x.strip().lower() == "normal" else 1)

# Divisão em Treino e Teste (usando os 25% reais para testar a matriz de confusão)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Ajusta a escala usando só o treino e aplica a mesma transformação no teste
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Salvando o scaler em disco para usar nos ataques adversariais depois
import joblib
os.makedirs("modelos_treinados", exist_ok=True)
joblib.dump(scaler, "modelos_treinados/scaler_binario.joblib")

# Converte os arrays para tensores, que são o formato que o PyTorch usa
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_raw.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_raw.values, dtype=torch.float32)

# Cria o DataLoader do conjunto de teste completo
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

N_FEATURES = X_train_scaled.shape[1]
print(f"Pipeline de dados concluído! Scaler salvo na pasta 'modelos_treinados'.")
print(f"Registros prontos para o Treino: {X_train_tensor.shape[0]}")
print(f"Registros guardados para o Teste Final: {X_test_tensor.shape[0]}")


Pipeline de dados concluído! Scaler salvo na pasta 'modelos_treinados'.
Registros prontos para o Treino: 112075
Registros guardados para o Teste Final: 37359


In [4]:
# Cada classe monta uma arquitetura usando os parâmetros salvos pelo tuning.

class MLP(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        camadas = []
        prev = input_dim
        n_camadas = config["hiperparametros"]["n_camadas"]
        ativacao = config["hiperparametros"]["ativacao"]
        
        for i in range(n_camadas):
            unidades = config["hiperparametros"][f"unidades_{i}"]
            dropout = config["hiperparametros"][f"dropout_{i}"]
            camadas.append(nn.Linear(prev, unidades))
            camadas.append(nn.ReLU() if ativacao == "relu" else nn.Tanh())
            if dropout > 0: camadas.append(nn.Dropout(dropout))
            prev = unidades
            
        camadas.append(nn.Linear(prev, 1))
        self.rede = nn.Sequential(*camadas)

    def forward(self, x): return self.rede(x)

# RNN, GRU e LSTM leem as colunas como uma sequência.
class Recorrente(nn.Module):
    def __init__(self, tipo_celula, input_dim, config):
        super().__init__()
        self.tipo_celula = tipo_celula
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        n_camadas = config["hiperparametros"]["n_camadas"]
        prev = input_dim
        
        for i in range(n_camadas):
            unidades = config["hiperparametros"][f"unidades_{i}"]
            dropout = config["hiperparametros"][f"dropout_{i}"]
            if tipo_celula == "RNN": camada = nn.RNN(prev, unidades, batch_first=True)
            elif tipo_celula == "GRU": camada = nn.GRU(prev, unidades, batch_first=True)
            else: camada = nn.LSTM(prev, unidades, batch_first=True)
            self.camadas.append(camada)
            self.dropouts.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            prev = unidades
            
        self.densa = nn.Linear(prev, 64)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(64, 1)

    def forward(self, x):
        for i, (camada, drop) in enumerate(zip(self.camadas, self.dropouts)):
            out, _ = camada(x)
            out = drop(out)
            if i < len(self.camadas) - 1: x = out
            else: x = out[:, -1, :]
        return self.saida(self.relu(self.densa(x)))

# A CNN também aplica filtros ao longo das colunas de entrada.
class CNN(nn.Module):
    def __init__(self, input_channels, config):
        super().__init__()
        camadas = []
        prev = input_channels
        
        n_blocos = config["hiperparametros"]["n_blocos_conv"]
        kernel_size = config["hiperparametros"]["kernel_size"]
        
        for i in range(n_blocos):
            filtros = config["hiperparametros"][f"filtros_{i}"]
            camadas.append(nn.Conv1d(prev, filtros, kernel_size=kernel_size, padding=kernel_size//2))
            camadas.append(nn.ReLU())
            camadas.append(nn.MaxPool1d(kernel_size=2, ceil_mode=True))
            prev = filtros
            
        self.conv = nn.Sequential(*camadas)
        self.pool_global = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(prev, 128)
        self.relu = nn.ReLU()
        
        self.dropout = nn.Dropout(config["hiperparametros"].get("dropout_final", 0.2))
        
        self.saida = nn.Linear(128, 1)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool_global(x).squeeze(-1)
        return self.saida(self.dropout(self.relu(self.densa(x))))

print("Classes das redes neurais carregadas com sucesso!")


Classes das redes neurais carregadas com sucesso!


In [5]:
# Separa os dados em lotes e define o treino final de cada modelo (os lotes são úteis pra não comer a memória toda da minha placa de vídeo)
BATCH_SIZE = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

EPOCHS = 50

def treinar_modelo_definitivo(nome, modelo_instancia, lr):
    print("\n" + "=" * 60)
    print(f"TREINAMENTO: {nome}")
    print("=" * 60)
    
    modelo_instancia = modelo_instancia.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(modelo_instancia.parameters(), lr=lr)
    
    for epoch in range(EPOCHS):
        modelo_instancia.train()
        # Mostra o andamento do treino nessa época com o tqdm.
        loop_treino = tqdm(train_loader, desc=f"Época {epoch+1}/{EPOCHS}", leave=False)
        
        for xb, yb in loop_treino:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if nome in ["RNN", "GRU", "LSTM"]: xb = xb.unsqueeze(-1)
            elif nome == "CNN": xb = xb.unsqueeze(-1).permute(0, 2, 1)
            
            optimizer.zero_grad()
            loss = criterion(modelo_instancia(xb).view(-1), yb)
            loss.backward()
            optimizer.step()
            loop_treino.set_postfix(loss=f"{loss.item():.4f}")
            
    # Salva os pesos e informações usadas depois para reconstruir a rede.
    caminho_salvamento = f"{DIR_MODELOS}/{nome}_binario.pt"
    torch.save({
        "state_dict": modelo_instancia.state_dict(),
        "n_classes": 2,
        "n_features": N_FEATURES
    }, caminho_salvamento)
    print(f"Cérebro neural {nome} salvo com sucesso em: {caminho_salvamento}")
    return modelo_instancia

# Monta cada rede com os parâmetros salvos pelo Optuna
instancias_modelos = {
    "MLP":  MLP(N_FEATURES, melhores_hp["MLP"]),
    "RNN":  Recorrente("RNN", 1, melhores_hp["RNN"]),
    "CNN":  CNN(1, melhores_hp["CNN"]),
    "GRU":  Recorrente("GRU", 1, melhores_hp["GRU"]),
    "LSTM": Recorrente("LSTM", 1, melhores_hp["LSTM"])
}

# Treina e guarda cada arquitetura
modelos_treinados = {}
for nome in ["MLP", "RNN", "CNN", "GRU", "LSTM"]:
    lr_modelo = melhores_hp[nome]["hiperparametros"]["learning_rate"]
    modelos_treinados[nome] = treinar_modelo_definitivo(nome, instancias_modelos[nome], lr_modelo)



TREINAMENTO: MLP


Cérebro neural MLP salvo com sucesso em: modelos_treinados/MLP_binario.pt

TREINAMENTO: RNN


Cérebro neural RNN salvo com sucesso em: modelos_treinados/RNN_binario.pt

TREINAMENTO: CNN


Cérebro neural CNN salvo com sucesso em: modelos_treinados/CNN_binario.pt

TREINAMENTO: GRU


Cérebro neural GRU salvo com sucesso em: modelos_treinados/GRU_binario.pt

TREINAMENTO: LSTM


Cérebro neural LSTM salvo com sucesso em: modelos_treinados/LSTM_binario.pt


In [6]:
# Avalia no teste e salva uma matriz de confusão para cada modelo

# Garante que uma pasta para os gráficos gerados exista no projeto e cria ela se não existir
DIR_GRAFICOS = "../graficos"
os.makedirs(DIR_GRAFICOS, exist_ok=True)

for nome, modelo in modelos_treinados.items():
    modelo.eval()
    predicoes_completas = []
    
    # Faz previsões em lotes para não carregar o teste inteiro na placa de vídeo e estourar a memória :(
    with torch.no_grad():
        for xb_t, _ in test_loader:
            xb_t = xb_t.to(DEVICE)
            if nome in ["RNN", "GRU", "LSTM"]: 
                xb_t = xb_t.unsqueeze(-1)
            elif nome == "CNN": 
                xb_t = xb_t.unsqueeze(-1).permute(0, 2, 1)
            
            logits = modelo(xb_t).view(-1)
            probs = torch.sigmoid(logits)
            # Usa 0,5 como limite entre Normal (0) e Ataque (1).
            preds_binarias = (probs > 0.5).int().cpu().numpy()
            predicoes_completas.append(preds_binarias)
            
    y_pred = np.concatenate(predicoes_completas)
    y_true = y_test_raw.values # Resgata o gabarito original do scikit-learn
    
    # Emissão dos relatórios no terminal
    print(f"\nRELATÓRIO ESTATÍSTICO DE DESEMPENHO: {nome}")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Ataque"]))
    
    # Cálculo matemático da Matriz de Confusão
    cm = confusion_matrix(y_true, y_pred)
    
    # Cada linha representa uma classe real, o que ajuda a comparar classes com tamanhos diferentes.
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Montagem da matriz de confusão usando o Seaborn e Matplotlib
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_norm, 
        annot=True,          # Adiciona os números em texto dentro dos quadrados
        fmt=".2f",           # Limita a exibição a duas casas decimais
        cmap="Blues",        # Paleta de cor azul pro binário
        xticklabels=["Normal", "Ataque"], 
        yticklabels=["Normal", "Ataque"]
    )
    
    plt.title(f"Matriz de Confusão - {nome} Binário")
    plt.xlabel("Previsão do Modelo (%)")
    plt.ylabel("Classe Real (%)")
    plt.tight_layout()
    
    # Salva cada gráfico de matriz no pc
    caminho_fig = os.path.join(DIR_GRAFICOS, f"classificacao_binaria_original_matriz_{nome}.png")
    plt.savefig(caminho_fig, dpi=150)
    plt.close() # Fecha a figura para economizar memória RAM do VS Code
    print(f"Gráfico de matriz exportado para: {caminho_fig}")



RELATÓRIO ESTATÍSTICO DE DESEMPENHO: MLP
              precision    recall  f1-score   support

      Normal       0.85      0.22      0.35      3361
      Ataque       0.93      1.00      0.96     33998

    accuracy                           0.93     37359
   macro avg       0.89      0.61      0.66     37359
weighted avg       0.92      0.93      0.91     37359

Gráfico de matriz exportado para: ../graficos/classificacao_binaria_original_matriz_MLP.png

RELATÓRIO ESTATÍSTICO DE DESEMPENHO: RNN
              precision    recall  f1-score   support

      Normal       0.82      0.21      0.34      3361
      Ataque       0.93      1.00      0.96     33998

    accuracy                           0.92     37359
   macro avg       0.87      0.60      0.65     37359
weighted avg       0.92      0.92      0.90     37359

Gráfico de matriz exportado para: ../graficos/classificacao_binaria_original_matriz_RNN.png

RELATÓRIO ESTATÍSTICO DE DESEMPENHO: CNN
              precision    recall  f

Com esses resultados, podemos ver que o modelo foi quase perfeito na identificação de ataques, mas, acabou tendo uma taxa muito elevada de falsos positivos. Assim como no multiclasse, vou remover as linhas duplicadas (que são inúmeras) e uma das features que é constante.

In [7]:
# Classificacao binaria com dados tratados: remove redundancias e mede o efeito sem substituir os resultados originais.
from sklearn.model_selection import GroupShuffleSplit

# Mantem a reproducibilidade; o limiar transforma probabilidades em Normal (0) ou Ataque (1).
SEED_TRATADO = 42
BATCH_SIZE_TRATADO = 256
EPOCHS_TRATADO = 50
LIMIAR_TRATADO = 0.5
DEVICE_TRATADO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED_TRATADO)
torch.manual_seed(SEED_TRATADO)

# Os resultados desta etapa ficam em uma pasta estavel e sao atualizados a cada nova execucao.
VERSAO_DADOS = "dados_tratados"
DIR_TRATADO = "classificacao_binaria_tratada"
DIR_MODELOS_TRATADO = os.path.join(DIR_TRATADO, "modelos")
DIR_GRAFICOS_TRATADO = os.path.join("..", "graficos")
os.makedirs(DIR_MODELOS_TRATADO, exist_ok=True)
os.makedirs(DIR_GRAFICOS_TRATADO, exist_ok=True)

# Reutiliza os hiperparametros do tuning binario ja realizado.
with open("tuning_bayesiano/resumo_melhores_hp_binario.json", "r") as arquivo:
    melhores_hp_tratado = json.load(arquivo)

df_original_exp = pd.read_csv("Normal+Attacks.csv")
linhas_originais_exp = len(df_original_exp)
contagem_original_exp = df_original_exp["label"].value_counts().to_dict()

# A feature constante nao traz variacao para aprender; ela e excluida somente neste tratado.
df_tratado = df_original_exp.drop(columns=["fwd_seg_size_min"], errors="ignore").copy()
X_todas_exp = df_tratado.drop(columns=["label"]).select_dtypes(include="number")
feature_names_exp = X_todas_exp.columns.tolist()

# Reduz todos os ataques a 1 e preserva Normal como 0.
y_todas_exp = df_tratado["label"].astype(str).str.strip().str.lower().ne("normal").astype("int64")
dados_exp = X_todas_exp.copy()
dados_exp["target"] = y_todas_exp.to_numpy()

# Remove repeticoes do mesmo vetor com o mesmo alvo; se um vetor tiver rotulos opostos, conserva ambos.
dados_exp = dados_exp.drop_duplicates().reset_index(drop=True)
linhas_apos_limpeza_exp = len(dados_exp)
X_tratado = dados_exp[feature_names_exp]
y_tratado = dados_exp["target"].to_numpy(dtype="float32")

# Agrupa vetores iguais para impedir que uma copia fique no treino e outra no teste.
# GroupShuffleSplit nao estratifica os rotulos.
grupos_exp = pd.util.hash_pandas_object(X_tratado, index=False).to_numpy()
splitter_exp = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED_TRATADO)
indices_treino_exp, indices_teste_exp = next(
    splitter_exp.split(X_tratado, y_tratado, groups=grupos_exp)
)
X_train_raw_exp = X_tratado.iloc[indices_treino_exp]
X_test_raw_exp = X_tratado.iloc[indices_teste_exp]
y_train_exp = y_tratado[indices_treino_exp]
y_test_exp = y_tratado[indices_teste_exp]

# Ajusta o scaler no treino e aplica os mesmos parametros ao teste.
scaler_tratado = StandardScaler()
X_train_exp = scaler_tratado.fit_transform(X_train_raw_exp)
X_test_exp = scaler_tratado.transform(X_test_raw_exp)
N_FEATURES_TRATADO = X_train_exp.shape[1]
X_train_tensor_exp = torch.tensor(X_train_exp, dtype=torch.float32)
X_test_tensor_exp = torch.tensor(X_test_exp, dtype=torch.float32)
y_train_tensor_exp = torch.tensor(y_train_exp, dtype=torch.float32)
y_test_tensor_exp = torch.tensor(y_test_exp, dtype=torch.float32)
train_loader_exp = DataLoader(
    TensorDataset(X_train_tensor_exp, y_train_tensor_exp),
    batch_size=BATCH_SIZE_TRATADO,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED_TRATADO),
)
test_loader_exp = DataLoader(
    TensorDataset(X_test_tensor_exp, y_test_tensor_exp), batch_size=1024, shuffle=False
)

class MLPBinarioTratado(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        hp = config["hiperparametros"]
        camadas = []
        anterior = input_dim
        for indice in range(hp["n_camadas"]):
            unidades = hp[f"unidades_{indice}"]
            dropout = hp[f"dropout_{indice}"]
            camadas.extend([nn.Linear(anterior, unidades), nn.ReLU() if hp["ativacao"] == "relu" else nn.Tanh()])
            if dropout > 0:
                camadas.append(nn.Dropout(dropout))
            anterior = unidades
        # Uma saida (logit) representa a probabilidade de Ataque.
        camadas.append(nn.Linear(anterior, 1))
        self.rede = nn.Sequential(*camadas)

    def forward(self, x):
        return self.rede(x)

class RecorrenteBinarioTratado(nn.Module):
    def __init__(self, tipo, config):
        super().__init__()
        hp = config["hiperparametros"]
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        anterior = 1
        for indice in range(hp["n_camadas"]):
            unidades = hp[f"unidades_{indice}"]
            dropout = hp[f"dropout_{indice}"]
            self.camadas.append({"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[tipo](anterior, unidades, batch_first=True))
            self.dropouts.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            anterior = unidades
        self.densa = nn.Linear(anterior, 64)
        self.saida = nn.Linear(64, 1)

    def forward(self, x):
        for indice, (camada, dropout) in enumerate(zip(self.camadas, self.dropouts)):
            saida, _ = camada(x)
            saida = dropout(saida)
            x = saida if indice < len(self.camadas) - 1 else saida[:, -1, :]
        return self.saida(torch.relu(self.densa(x)))

class CNNBinarioTratado(nn.Module):
    def __init__(self, config):
        super().__init__()
        hp = config["hiperparametros"]
        camadas = []
        anterior = 1
        kernel_size = hp["kernel_size"]
        for indice in range(hp["n_blocos_conv"]):
            filtros = hp[f"filtros_{indice}"]
            camadas.extend([nn.Conv1d(anterior, filtros, kernel_size, padding=kernel_size // 2), nn.ReLU(), nn.MaxPool1d(2, ceil_mode=True)])
            anterior = filtros
        self.conv = nn.Sequential(*camadas)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(anterior, 128)
        self.dropout = nn.Dropout(hp.get("dropout_final", 0.2))
        self.saida = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(self.conv(x)).squeeze(-1)
        return self.saida(self.dropout(torch.relu(self.densa(x))))


# Converte os tensores para a forma de entrada esperada pelas redes recorrentes e pela CNN.
def preparar_entrada_binaria(nome, xb):
    if nome in ("RNN", "GRU", "LSTM"):
        return xb.unsqueeze(-1)
    if nome == "CNN":
        return xb.unsqueeze(-1).permute(0, 2, 1)
    return xb


construtores_exp = {
    "MLP": lambda: MLPBinarioTratado(N_FEATURES_TRATADO, melhores_hp_tratado["MLP"]),
    "RNN": lambda: RecorrenteBinarioTratado("RNN", melhores_hp_tratado["RNN"]),
    "CNN": lambda: CNNBinarioTratado(melhores_hp_tratado["CNN"]),
    "GRU": lambda: RecorrenteBinarioTratado("GRU", melhores_hp_tratado["GRU"]),
    "LSTM": lambda: RecorrenteBinarioTratado("LSTM", melhores_hp_tratado["LSTM"]),
}

# Guarda o scaler junto dos pesos para repetir o mesmo pre-processamento na inferencia.
joblib.dump(scaler_tratado, os.path.join(DIR_MODELOS_TRATADO, "scaler_binario.joblib"))
metricas_treinamento = {}

for nome, construir_modelo in construtores_exp.items():
    print(f"\nTreinando {nome} com dados tratados (binario) ({DEVICE_TRATADO})")
    modelo = construir_modelo().to(DEVICE_TRATADO)
    # A funcao combina sigmoid e entropia cruzada para treino com um unico logit binario.
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(modelo.parameters(), lr=melhores_hp_tratado[nome]["hiperparametros"]["learning_rate"])

    for epoca in range(EPOCHS_TRATADO):
        modelo.train()
        for xb, yb in tqdm(train_loader_exp, desc=f"{nome} {epoca + 1}/{EPOCHS_TRATADO}", leave=False):
            xb, yb = xb.to(DEVICE_TRATADO), yb.to(DEVICE_TRATADO)
            xb = preparar_entrada_binaria(nome, xb)
            optimizer.zero_grad()
            loss = criterion(modelo(xb).view(-1), yb)
            loss.backward()
            optimizer.step()

    # Calcula probabilidades no teste; o limiar escolhido controla o equilibrio entre alarmes falsos e ataques perdidos.
    modelo.eval()
    probabilidades_exp = []
    with torch.no_grad():
        for xb, _ in test_loader_exp:
            xb = preparar_entrada_binaria(nome, xb.to(DEVICE_TRATADO))
            probabilidades_exp.extend(torch.sigmoid(modelo(xb).view(-1)).cpu().numpy().tolist())

    y_prob_exp = np.asarray(probabilidades_exp)
    y_pred_exp = (y_prob_exp >= LIMIAR_TRATADO).astype("int64")
    y_true_exp = y_test_exp.astype("int64")
    relatorio_exp = classification_report(y_true_exp, y_pred_exp, labels=[0, 1], target_names=["Normal", "Ataque"], output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true_exp, y_pred_exp, labels=[0, 1]).ravel()
    # FPR mede quantos Normais viraram alarmes; recall mede quantos Ataques foram detectados.
    metricas_treinamento[nome] = {
        "threshold": LIMIAR_TRATADO,
        "accuracy": float(accuracy_score(y_true_exp, y_pred_exp)),
        "false_positive_rate": float(fp / (fp + tn)) if fp + tn else 0.0,
        "specificity": float(tn / (tn + fp)) if tn + fp else 0.0,
        "attack_recall": float(tp / (tp + fn)) if tp + fn else 0.0,
        "attack_precision": float(tp / (tp + fp)) if tp + fp else 0.0,
        "classification_report": relatorio_exp,
    }
    print(classification_report(y_true_exp, y_pred_exp, labels=[0, 1], target_names=["Normal", "Ataque"], zero_division=0))
    print(f"Taxa de falsos positivos entre Normais: {metricas_treinamento[nome]['false_positive_rate']:.2%}")

    # Inclui nomes e ordem das features e limiar para permitir carregar este modelo sem ambiguidade.
    torch.save({
        "state_dict": modelo.state_dict(),
        "n_classes": 2,
        "n_features": N_FEATURES_TRATADO,
        "feature_names": feature_names_exp,
        "threshold": LIMIAR_TRATADO,
    }, os.path.join(DIR_MODELOS_TRATADO, f"{nome}_binario.pt"))

    # Cada linha da matriz representa uma classe real; normalizar destaca FPR e recall apesar do desbalanceamento.
    matriz_exp = confusion_matrix(y_true_exp, y_pred_exp, labels=[0, 1])
    matriz_normalizada_exp = matriz_exp.astype(float) / np.maximum(matriz_exp.sum(axis=1, keepdims=True), 1) * 100
    plt.figure(figsize=(6, 5))
    sns.heatmap(matriz_normalizada_exp, annot=True, fmt=".2f", cmap="Blues", xticklabels=["Normal", "Ataque"], yticklabels=["Normal", "Ataque"])
    plt.title(f"Matriz de Confusao - {nome} Binario (deduplicado)")
    plt.xlabel("Previsao do modelo (%)")
    plt.ylabel("Classe real (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_GRAFICOS_TRATADO, f"classificacao_binaria_tratada_matriz_{nome}.png"), dpi=150)
    plt.close()

# Registra as metricas completas e um resumo para comparar execucoes e arquiteturas.
with open(os.path.join(DIR_TRATADO, "metricas_treinamento.json"), "w", encoding="utf-8") as arquivo:
    json.dump(metricas_treinamento, arquivo, indent=2, ensure_ascii=False)

resumo_treinamento = {
    "versao": VERSAO_DADOS,
    "seed": SEED_TRATADO,
    "epocas": EPOCHS_TRATADO,
    "threshold": LIMIAR_TRATADO,
    "linhas_originais": linhas_originais_exp,
    "duplicatas_removidas": linhas_originais_exp - linhas_apos_limpeza_exp,
    "linhas_apos_limpeza": linhas_apos_limpeza_exp,
    "vetores_de_features_com_rotulos_conflitantes": int((dados_exp.groupby(feature_names_exp, dropna=False)["target"].nunique() > 1).sum()),
    "coluna_removida": "fwd_seg_size_min",
    "contagem_binaria_apos_limpeza": {"Normal": int((y_tratado == 0).sum()), "Ataque": int((y_tratado == 1).sum())},
    "features": feature_names_exp,
    "linhas_treino": int(len(y_train_exp)),
    "linhas_teste": int(len(y_test_exp)),
    "metricas": {nome: {chave: valores[chave] for chave in ("accuracy", "false_positive_rate", "specificity", "attack_recall", "attack_precision")} for nome, valores in metricas_treinamento.items()},
}
with open(os.path.join(DIR_TRATADO, "resumo_treinamento.json"), "w", encoding="utf-8") as arquivo:
    json.dump(resumo_treinamento, arquivo, indent=2, ensure_ascii=False)

print(f"\nClassificacao binaria concluida. Resultados atuais em: {DIR_TRATADO}")


Treinando MLP com dados tratados (binario) (cuda)


              precision    recall  f1-score   support

      Normal       0.49      0.87      0.63       134
      Ataque       1.00      0.99      0.99      8777

    accuracy                           0.98      8911
   macro avg       0.74      0.93      0.81      8911
weighted avg       0.99      0.98      0.99      8911

Taxa de falsos positivos entre Normais: 12.69%

Treinando RNN com dados tratados (binario) (cuda)


              precision    recall  f1-score   support

      Normal       0.64      0.10      0.18       134
      Ataque       0.99      1.00      0.99      8777

    accuracy                           0.99      8911
   macro avg       0.81      0.55      0.59      8911
weighted avg       0.98      0.99      0.98      8911

Taxa de falsos positivos entre Normais: 89.55%

Treinando CNN com dados tratados (binario) (cuda)


              precision    recall  f1-score   support

      Normal       0.55      0.54      0.54       134
      Ataque       0.99      0.99      0.99      8777

    accuracy                           0.99      8911
   macro avg       0.77      0.77      0.77      8911
weighted avg       0.99      0.99      0.99      8911

Taxa de falsos positivos entre Normais: 46.27%

Treinando GRU com dados tratados (binario) (cuda)


              precision    recall  f1-score   support

      Normal       1.00      0.04      0.09       134
      Ataque       0.99      1.00      0.99      8777

    accuracy                           0.99      8911
   macro avg       0.99      0.52      0.54      8911
weighted avg       0.99      0.99      0.98      8911

Taxa de falsos positivos entre Normais: 95.52%

Treinando LSTM com dados tratados (binario) (cuda)


              precision    recall  f1-score   support

      Normal       1.00      0.04      0.09       134
      Ataque       0.99      1.00      0.99      8777

    accuracy                           0.99      8911
   macro avg       0.99      0.52      0.54      8911
weighted avg       0.99      0.99      0.98      8911

Taxa de falsos positivos entre Normais: 95.52%

Classificacao binaria concluida. Resultados atuais em: classificacao_binaria_tratada
